In [2]:
import pandas as pd
import numpy as np

fy2021 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/LCA_Disclosure_Data_FY2021_Q4.xlsx"

df = pd.read_excel(fy2021)

print(df.shape)
print(df.head())

(126576, 96)
          CASE_NUMBER CASE_STATUS RECEIVED_DATE DECISION_DATE  \
0  I-200-21175-424928   Certified    2021-06-24    2021-07-01   
1  I-200-21175-424959   Certified    2021-06-24    2021-07-01   
2  I-200-21175-425244   Certified    2021-06-24    2021-07-01   
3  I-200-21175-423449   Certified    2021-06-24    2021-07-01   
4  I-200-21175-425533   Certified    2021-06-24    2021-07-01   

  ORIGINAL_CERT_DATE VISA_CLASS                          JOB_TITLE  \
0                NaT       H-1B                Software Engineer 3   
1                NaT       H-1B                    MANAGER, STRESS   
2                NaT       H-1B                       Manager JC50   
3                NaT       H-1B  Investment Banking Senior Analyst   
4                NaT       H-1B                  Software Engineer   

     SOC_CODE                              SOC_TITLE FULL_TIME_POSITION  ...  \
0  15-1132.00      Software Developers, Applications                  Y  ...   
1  17-2011.00  

In [4]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'RECEIVED_DATE', 'DECISION_DATE', 'ORIGINAL_CERT_DATE', 'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE', 'FULL_TIME_POSITION', 'BEGIN_DATE', 'END_DATE', 'TOTAL_WORKER_POSITIONS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'EMPLOYER_NAME', 'TRADE_NAME_DBA', 'EMPLOYER_ADDRESS1', 'EMPLOYER_ADDRESS2', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'NAICS_CODE', 'EMPLOYER_POC_LAST_NAME', 'EMPLOYER_POC_FIRST_NAME', 'EMPLOYER_POC_MIDDLE_NAME', 'EMPLOYER_POC_JOB_TITLE', 'EMPLOYER_POC_ADDRESS1', 'EMPLOYER_POC_ADDRESS2', 'EMPLOYER_POC_CITY', 'EMPLOYER_POC_STATE', 'EMPLOYER_POC_POSTAL_CODE', 'EMPLOYER_POC_COUNTRY', 'EMPLOYER_POC_PROVINCE', 'EMPLOYER_POC_PHONE', 'EMPLOYER_POC_PHONE_EXT', 'EMPLOYER_POC_EMAIL', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_LAST_NAME', 'AGENT_ATT

In [6]:
import pandas as pd

df = pd.read_excel(fy2021)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
AMAZON.COM SERVICES LLC                       2499
MICROSOFT CORPORATION                         1838
GOOGLE LLC                                    1686
FACEBOOK, INC.                                1243
APPLE INC.                                     988
AMAZON WEB SERVICES, INC.                      690
AMAZON DEVELOPMENT CENTER U.S., INC.           288
TESLA, INC.                                    229
FACEBOOK INC.                                  218
NVIDIA CORPORATION                             113
ADVANCED MICRO DEVICES, INC.                   107
AMAZON DATA SERVICES, INC.                     104
AMAZON WEB SERVICES INC                         86
FACEBOOK, INC                                   63
MICRON TECHNOLOGY, INC.                         52
AMAZON ADVERTISING LLC                          40
AMAZON DATA SERVICES, INC                       34
MICRONET IT SOLUTIONS, INC.                     12
APPLET SYSTEMS LLC                               7
METASENSE, INC.  

In [8]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Amazon       2499
Microsoft    1838
Alphabet     1686
Meta         1243
Apple         988
Tesla         229
NVIDIA        113
AMD           107
Micron         52
Name: count, dtype: int64


In [10]:
df_clean = df.copy()

In [12]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Seattle     1302
New York     432
Austin       422
San Jose      36
Dallas        35
Name: count, dtype: int64


In [14]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year     2222
Hour        4
Month       1
Name: count, dtype: int64


In [16]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

    Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  Annual_Wage
104  Amazon   Seattle               137800.0             YEAR     137800.0
110    Meta   Seattle               145000.0             YEAR     145000.0
128  Amazon  New York               127000.0             YEAR     127000.0
286  Amazon   Seattle               100000.0             YEAR     100000.0
394  Amazon   Seattle               105000.0             YEAR     105000.0
438   Apple    Austin               170000.0             YEAR     170000.0
620  Amazon   Seattle               155000.0             YEAR     155000.0
640    Meta   Seattle               203000.0             YEAR     203000.0
660  Amazon   Seattle               123000.0             YEAR     123000.0
678  Amazon   Seattle               112000.0             YEAR     112000.0


In [18]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    2.227000e+03
mean     3.313528e+05
std      9.043120e+06
min      4.417900e+04
25%      1.150000e+05
50%      1.432800e+05
75%      1.600000e+05
max      4.268888e+08
Name: Annual_Wage, dtype: float64


In [20]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [22]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage     Mean_Wage
0     Austin        AMD                 55    134118.00  1.363148e+05
1     Austin   Alphabet                 39    140000.00  1.403203e+05
2     Austin     Amazon                142    122500.00  1.163518e+05
3     Austin      Apple                118    155250.00  1.564442e+05
4     Austin       Meta                 21    150000.00  2.047255e+07
5     Austin  Microsoft                 13    150000.00  1.511854e+05
6     Austin     NVIDIA                  8    157394.00  1.641530e+05
7     Austin      Tesla                 26    129000.00  1.364313e+05
8     Dallas     Amazon                 32     99000.00  1.059385e+05
9     Dallas  Microsoft                  3     98600.00  1.036200e+05
10  New York   Alphabet                146    161000.00  1.651958e+05
11  New York     Amazon                122    125485.00  1.257399e+05
12  New York      Apple                  9    159418.00  1.654437e+05
13  New York       M

In [30]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_TITLE",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2021.csv",
    index=False
)